# Notebook for clustering.

Run scripts as interactive notebooks in the follwing order:\
    preprocessing.py  ->  plot_corr_assoc.py  ->  dimensionality_reduction.py  ->  clustering.ipynb

In [ ]:
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotnine as p9
from numpy.typing import NDArray
from sklearn.cluster import DBSCAN, HDBSCAN, OPTICS, KMeans  # type: ignore
from sklearn.metrics import silhouette_samples, silhouette_score

from dimensionality_reduction import plot_reduced_data
from utils import check_save_path, load_data, save_data

pd.options.mode.copy_on_write = True


type ClusteringMethod = Literal["KMeans", "DBSCAN", "OPTICS", "HDBSCAN"]
type Clustering = KMeans | DBSCAN | OPTICS | HDBSCAN


def plot_silhouette(data: pd.DataFrame, cluster_labels: pd.Series) -> p9.ggplot:
    # Compute silhouettes
    silhouettes = silhouette_samples(data, cluster_labels, metric="euclidean")

    # Create ggplot data
    data_ggplot = pd.DataFrame({"cluster": cluster_labels, "silhouette": silhouettes}).assign(
        **{"rank": lambda x: x.groupby("cluster")["silhouette"].rank(method="dense", ascending=False)}
    )

    # Create silhouette plot
    silhouette_plot = (
        p9.ggplot(data_ggplot, p9.aes(x="factor(rank)", y="silhouette", fill="cluster"))
        + p9.facet_grid(rows="cluster", cols=None, scales="free_y", space="free_y")
        + p9.geom_bar(stat="identity", width=1)
        + p9.coord_flip()
        + p9.labs(x="")
        + p9.theme_538()
        + p9.theme(
            axis_text_y=p9.element_blank(),
            axis_ticks_y=p9.element_blank(),
            panel_grid=p9.element_blank(),
            strip_text=p9.element_blank(),
            panel_grid_major_x=p9.element_line(color="gray", alpha=0.5),
            panel_spacing=0,
            # panel_border=p9.element_rect(color="gray", alpha=0.5),
            plot_background=p9.element_rect(fill="gray", alpha=0.3),
        )
    )

    return silhouette_plot


In [ ]:
# Create RNG for reproducibility
rng = np.random.default_rng(394325)

# Hardcoded paths, change to apply to new data
data_reduced_path = Path("outputs/dim_red/esa_2018_2021_reduced.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")

image_folder = "outputs/images"
plot_save_path = Path(image_folder, "clustering_plot.png")

allow_overwrite = True

# Read in data
data_reduced: pd.DataFrame = load_data(data_reduced_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

# # Normalize data
# from sklearn.preprocessing import StandardScaler
# ...

# from scipy.spatial.distance import pdist, squareform
# distances = pdist(data_reduced.values, metric='euclidean')
# dist_matrix = squareform(distances)
# ...

### k-means clustering with silhouette plots

In [ ]:
n_clusters_list = [4, 5, 6, 7]
clustering_list = []
cluster_labels_list = []
silhouette_plot_list = []
for n_clusters in n_clusters_list:
    clustering = KMeans(n_clusters=n_clusters, random_state=np.random.RandomState(rng.bit_generator))
    cluster_labels = pd.Series(clustering.fit_predict(data_reduced)).replace(-1, pd.NA).astype("category")
    # Print information about clustering
    print(
        f"{len(cluster_labels.cat.categories.values)} clusters, {100 * cluster_labels.isna().mean():0.2f}% outliers"
    )
    print(
        f"Percentage in clusters: {(
            cluster_labels
            .value_counts(normalize=True, dropna=False)
            .apply(lambda x: 100 * x)
            .sort_index()
            .to_dict()
        )}"
    )
    # Silhouette plot
    silhouette_plot = plot_silhouette(data_reduced, cluster_labels)
    # Save
    clustering_list.append(clustering)
    cluster_labels_list.append(cluster_labels)
    silhouette_plot_list.append(silhouette_plot)

p1, p2, p3, p4 = silhouette_plot_list
silhouette_plots = (p1 | p2) / (p3 | p4)
silhouette_plots

### DBSCAN clustering

In [ ]:
clustering = DBSCAN(eps=6.5, min_samples = 10, n_jobs=-1)

clustering.fit(data_reduced)
cluster_labels = pd.Series(clustering.labels_).replace(-1, pd.NA).astype("category")

### HDBSCAN clustering

In [ ]:
clustering = HDBSCAN(min_cluster_size=10, store_centers="both", n_jobs=-1)

clustering.fit(data_reduced)
cluster_labels = pd.Series(clustering.labels_).replace(-1, pd.NA).astype("category")
cluster_probs = clustering.probabilities_
cluster_centroids = clustering.centroids_
cluster_medoids = clustering.medoids_

### OPTICS clustering

In [ ]:
clustering = OPTICS(n_jobs=-1)

clustering.fit(data_reduced)
cluster_labels = pd.Series(clustering.labels_).replace(-1, pd.NA).astype("category")